In [ ]:
# 1. 固定入口：挂载 Drive，并读取唯一可变的测试请求 JSON。
from google.colab import drive
drive.mount('/content/drive')

import json
import os
from pathlib import Path
import re
import subprocess
import sys

DRIVE_PROJECT_ROOT = os.environ.get('SSTW_DRIVE_PROJECT_ROOT', '/content/drive/MyDrive/SSTW')
REQUEST_PATH = Path(os.environ.get(
    'SSTW_COLAB_TEST_REQUEST',
    f'{DRIVE_PROJECT_ROOT}/requests/colab_test_request.json',
))
if not REQUEST_PATH.is_file():
    raise FileNotFoundError(
        f'Missing Colab test request: {REQUEST_PATH}. Copy and edit '
        'configs/paper_workflow/colab_test_request_example.json first.'
    )
request = json.loads(REQUEST_PATH.read_text(encoding='utf-8-sig'))
repository = request.get('repository')
if not isinstance(repository, dict):
    raise TypeError('request.repository must be a JSON object')
REPO_URL = str(repository.get('url') or '').strip()
REPO_REF = str(repository.get('ref') or '').strip()
if REPO_URL != 'https://github.com/RICHAAARC/SSTW.git':
    raise ValueError('request.repository.url is not the governed SSTW repository')
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9._/-]*', REPO_REF) or '..' in REPO_REF:
    raise ValueError('request.repository.ref is not a safe Git revision')
REPO_DIR = '/content/SSTW'
MODEL_CACHE_ROOT = '/content/SSTW_model_cache'
os.environ['HF_HOME'] = MODEL_CACHE_ROOT
os.environ['HF_HUB_CACHE'] = f'{MODEL_CACHE_ROOT}/hub'
print('Drive project root =', DRIVE_PROJECT_ROOT)
print('Ephemeral model cache =', MODEL_CACHE_ROOT)
print('Request path =', REQUEST_PATH)
print('Requested test =', request.get('test_id'), request.get('parameters', {}).get('phase'))
print(subprocess.getoutput('nvidia-smi'))


# SSTW stable Colab test runner

该 Notebook 是固定薄入口。后续测试通常只编辑 Google Drive 中的 `SSTW/requests/colab_test_request.json`；仓库白名单 runner 负责输入校验、本地 GPU 热路径执行，以及把 zip 和 manifest 写回 `SSTW/diagnostic_tests/`。诊断结果不属于论文证据。


In [ ]:
# 2. 按请求获取仓库，并冻结本次实际执行 commit。
repo_path = Path(REPO_DIR)
if not repo_path.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, REPO_DIR], check=True)
elif not (repo_path / '.git').is_dir():
    raise RuntimeError(f'Existing path is not a Git repository: {repo_path}')
subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'fetch', '--depth=1', 'origin', REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
RESOLVED_REPO_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
print('SSTW repository commit =', RESOLVED_REPO_COMMIT)


In [ ]:
# 3. 使用 Colab 原生 torch/CUDA/NumPy；仅在能力不足时补齐兼容依赖。
RUNTIME_COMPATIBILITY_DECISION = Path('/content/sstw_colab_test_runtime_compatibility.json')
bootstrap_command = [
    sys.executable,
    'scripts/bootstrap_colab_test_runtime.py',
    '--decision-output', str(RUNTIME_COMPATIBILITY_DECISION),
]
bootstrap_result = subprocess.run(bootstrap_command, check=False)
if bootstrap_result.returncode != 0:
    raise RuntimeError(f'Colab runtime compatibility bootstrap failed, return_code={bootstrap_result.returncode}')
runtime_compatibility = json.loads(
    RUNTIME_COMPATIBILITY_DECISION.read_text(encoding='utf-8')
)
print('Runtime source =', runtime_compatibility['runtime_source'])
print('Compatibility install executed =', runtime_compatibility['compatibility_install_executed'])
print('Protected core versions =', runtime_compatibility['protected_core_versions'])


In [ ]:
# 4. 认证只从 Colab secret / Drive 私有文件引导，不显示凭据。
from paper_workflow.colab_utils.huggingface_authentication import bootstrap_huggingface_authentication
from paper_workflow.colab_utils.trajectory_authentication import load_trajectory_authentication_from_private_drive

huggingface_authentication = bootstrap_huggingface_authentication(
    DRIVE_PROJECT_ROOT,
    environ=os.environ,
)
trajectory_authentication = load_trajectory_authentication_from_private_drive(
    DRIVE_PROJECT_ROOT,
    environ=os.environ,
)
print('Hugging Face authentication =', huggingface_authentication)
print('trajectory authentication =', trajectory_authentication)


In [ ]:
# 5. 只调用仓库服务器 CLI；测试分派和结果写入都在仓库模块中完成。
from workflows.streaming_command import run_streaming_command

WORKFLOW_PROFILE = 'colab_test'
NOTEBOOK_ROLE = 'colab_test'
SERVER_PIPELINE = 'colab_test'
DECISION_OUTPUT = Path('/content/sstw_colab_test_decision.json')
server_command = [
    sys.executable,
    'scripts/run_generative_video_server_workflow.py',
    '--project-root', DRIVE_PROJECT_ROOT,
    '--repo-root', REPO_DIR,
    '--workflow-profile', WORKFLOW_PROFILE,
    '--pipeline', SERVER_PIPELINE,
    '--colab-test-request-path', str(REQUEST_PATH),
    '--local-workspace-root', '/content/SSTW_colab_test_workspace',
    '--local-package-cache-root', '/content/SSTW_colab_test_packages',
    '--decision-output', str(DECISION_OUTPUT),
]
print('server workflow pipeline =', SERVER_PIPELINE)
DECISION_OUTPUT.unlink(missing_ok=True)
result = run_streaming_command(server_command)
if result.returncode != 0:
    raise RuntimeError(f'Colab test failed, return_code={result.returncode}')
decision = json.loads(DECISION_OUTPUT.read_text(encoding='utf-8'))
pipeline_result = decision['pipeline_results'][0]
drive_result_zip = Path(pipeline_result['drive_result_zip'])
drive_result_manifest = Path(pipeline_result['drive_result_manifest'])
drive_root = Path(DRIVE_PROJECT_ROOT).resolve()
for result_path in (drive_result_zip, drive_result_manifest):
    if not result_path.is_file():
        raise FileNotFoundError(f'Expected Drive result is missing: {result_path}')
    result_path.resolve().relative_to(drive_root)
print('server workflow decision =', decision['server_workflow_decision'])
print('Google Drive result zip =', drive_result_zip)
print('Google Drive manifest =', drive_result_manifest)


## 故障恢复打包（仅在 Cell 5 中断后手动运行）

下面的 Cell 不重跑 GPU 任务，只把 `/content` 中已产生的局部 records、checkpoint、validation 和失败 decision 打成单个排障 ZIP，并与最小 manifest 写入 Drive 的 `SSTW/diagnostic_tests/<test_id>/recovery/`。恢复包不是正式结果，不能用于阶段推进或论文证据。


In [ ]:
# 6. Cell 5 失败后单独运行；保存 /content 局部产物用于排障。
from workflows.streaming_command import run_streaming_command

RECOVERY_DECISION_OUTPUT = Path('/content/sstw_colab_test_recovery_decision.json')
recovery_command = [
    sys.executable,
    'scripts/run_generative_video_server_workflow.py',
    '--project-root', DRIVE_PROJECT_ROOT,
    '--repo-root', REPO_DIR,
    '--workflow-profile', 'colab_test',
    '--pipeline', 'colab_test',
    '--colab-test-request-path', str(REQUEST_PATH),
    '--local-workspace-root', '/content/SSTW_colab_test_workspace',
    '--local-package-cache-root', '/content/SSTW_colab_test_packages',
    '--package-colab-test-recovery',
    '--colab-test-local-runtime-root', '/content',
    '--decision-output', str(RECOVERY_DECISION_OUTPUT),
]
if DECISION_OUTPUT.is_file():
    recovery_command.extend([
        '--colab-test-run-decision-path', str(DECISION_OUTPUT),
    ])
recovery_result = run_streaming_command(recovery_command)
if recovery_result.returncode != 0:
    raise RuntimeError(f'Colab recovery packaging failed, return_code={recovery_result.returncode}')
recovery_decision = json.loads(RECOVERY_DECISION_OUTPUT.read_text(encoding='utf-8'))
if recovery_decision.get('formal_result') is not False:
    raise RuntimeError('Recovery package must remain non-formal')
print('Recovery ZIP =', recovery_decision['drive_result_zip'])
print('Recovery manifest =', recovery_decision['drive_result_manifest'])
print('Claim boundary =', recovery_decision['claim_support_status'])
